<a href="https://colab.research.google.com/github/edgi-govdata-archiving/GHG-CDP/blob/main/Census_Lookup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas

# Load census blocks data
cbs = pandas.read_csv("/content/drive/Shareddrives/EDGI - Shared NEW/12_Environmental_Enforcement_Watch/09_Data/US_Census_Blocks_v1_4586937457682438194.csv", dtype={"Full Census Block Geographic Identifier":str},  usecols=["Full Census Block Geographic Identifier", "Decennial Population Count"]) # Only need these two columns
cbs

,Full Census Block Geographic Identifier,Decennial Population Count
0,060990038022002,78
1,060990038044007,141
2,060990004031009,99
3,060990009091006,53
4,060990009082013,30
...,...,...
8180861,510190305033032,0
8180862,510190302021043,0
8180863,510190303001027,6
8180864,510190301013007,12


In [ ]:
# Create block group code
cbs["BlockGroup"] = cbs['Full Census Block Geographic Identifier'].str[0:12] # Block to block group, 15->12
cbs

,Full Census Block Geographic Identifier,Decennial Population Count,BlockGroup
0,060990038022002,78,060990038022
1,060990038044007,141,060990038044
2,060990004031009,99,060990004031
3,060990009091006,53,060990009091
4,060990009082013,30,060990009082
...,...,...,...
8180861,510190305033032,0,510190305033
8180862,510190302021043,0,510190302021
8180863,510190303001027,6,510190303001
8180864,510190301013007,12,510190301013


In [ ]:
# Get CD-Block lookup table
# Downloaded from https://www.census.gov/geographies/mapping-files/2025/dec/rdo/119-congressional-district-bef.html on July 14 2026
url = "https://www2.census.gov/programs-surveys/decennial/rdo/mapping-files/2025/119-congressional-district-befs/cd119.zip"
import os
from urllib.request import urlretrieve
from zipfile import ZipFile

zip_filename = "cd119.zip"
extract_path = "./extracted_files"

# Step 1: Download and save the ZIP file locally
urlretrieve(url, zip_filename)

# Step 2: Unzip the local file
with ZipFile(zip_filename, "r") as zip_file:
    zip_file.extractall(extract_path)

In [ ]:
# Load lookup table
import pandas
lu = pandas.read_csv("/content/extracted_files/NationalCD119.txt", dtype={"GEOID": str, "CDFP": str}) # str to preserve leading zeros
lu["GEOID_CD"] = lu["GEOID"].str[:2] + "" + lu["CDFP"]
lu

,GEOID,CDFP,GEOID_CD
0,020130001001000,00,0200
1,020130001001001,00,0200
2,020130001001002,00,0200
3,020130001001003,00,0200
4,020130001001004,00,0200
...,...,...,...
8174950,371999604002130,11,3711
8174951,371999604002131,11,3711
8174952,371999604002132,11,3711
8174953,371999604002133,11,3711


In [ ]:
# Join CD-Block LU to Census block pop counts
results = cbs.set_index("Full Census Block Geographic Identifier").join(lu.set_index("GEOID"), how="left")
results

,Decennial Population Count,BlockGroup,CDFP,GEOID_CD
Full Census Block Geographic Identifier,,,,
060990038022002,78,060990038022,13,0613
060990038044007,141,060990038044,13,0613
060990004031009,99,060990004031,05,0605
060990009091006,53,060990009091,05,0605
060990009082013,30,060990009082,05,0605
...,...,...,...,...
510190305033032,0,510190305033,09,5109
510190302021043,0,510190302021,05,5105
510190303001027,6,510190303001,09,5109


In [ ]:
# Test (population counts)
pops = results.groupby(by="GEOID_CD")[["Decennial Population Count"]].sum()
pops

,Decennial Population Count
GEOID_CD,
0101,717754
0102,717754
0103,717754
0104,717754
0105,717754
...,...
5506,736714
5507,736715
5508,736714
